# Easy Agent — Core Agentic Loop

A self-contained Python walkthrough of the **Reason → Act → Observe** loop that powers Easy Agent.
Every class and function is inlined from the TypeScript source — no project imports are used.

| Notebook section | TypeScript source |
|---|---|
| 2. Data models | `src/types/message.ts` |
| 3. Permission system | `src/permissions/permissions.ts` |
| 4. Token budget | `src/context/autoCompact.ts` |
| 5. Streaming | `src/services/api/streaming.ts` |
| 6. Tool registry (stub) | `src/tools/index.ts` |
| 7. Tool execution pipeline | `src/core/agenticLoop.ts` → `runTools()` |
| 8. Core `query()` loop | `src/core/agenticLoop.ts` → `query()` |
| 9. `QueryEngine` | `src/core/queryEngine.ts` |
| 10–11. Live demos | — |

> Real Anthropic API calls require `ANTHROPIC_API_KEY` in the environment.

## 1. Imports, Environment & Path Discovery

In [ ]:
import os, json, re, asyncio
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, AsyncGenerator, Literal, Optional, Union

from dotenv import load_dotenv
import anthropic

# ── Path discovery ────────────────────────────────────────────────────────
_cwd = Path('.').resolve()
PROJECT_ROOT = _cwd
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / 'pyproject.toml').exists() or (PROJECT_ROOT / 'package.json').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

for _env_candidate in [PROJECT_ROOT / '.env', PROJECT_ROOT.parent / '.env']:
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        print(f'Loaded .env from {_env_candidate}')
        break

ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
print('ANTHROPIC_API_KEY:', 'found' if ANTHROPIC_API_KEY else 'NOT SET — live cells will be skipped')

## 2. Core Data Models

Inlined from `src/types/message.ts`.
These types flow through every layer — the streaming layer assembles them,
the loop threads them between turns, and the tool pipeline populates them.

In [ ]:
@dataclass
class TextBlock:
    type: str = 'text'
    text: str = ''

@dataclass
class ToolUseBlock:
    type: str = 'tool_use'
    id: str = ''        # e.g. 'toolu_01ABC...'
    name: str = ''      # e.g. 'Read', 'Bash'
    input: dict = field(default_factory=dict)

@dataclass
class ToolResultBlock:
    type: str = 'tool_result'
    tool_use_id: str = ''
    content: str = ''
    is_error: bool = False

ContentBlock = Union[TextBlock, ToolUseBlock, ToolResultBlock]

@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    cache_creation_input_tokens: Optional[int] = None
    cache_read_input_tokens: Optional[int] = None

@dataclass
class ToolResult:
    content: str = ''
    is_error: bool = False

@dataclass
class MessageParam:
    role: str         # 'user' | 'assistant'
    content: Any      # str or list[ContentBlock]

@dataclass
class LoopState:
    messages: list = field(default_factory=list)
    turn_count: int = 0
    aborted: bool = False

print('Data models defined.')

## 3. Permission System

Inlined from `src/permissions/permissions.ts`.
Every tool call passes through `check_permission()` before execution.

**Decision tree** (in order):
1. `auto` mode → always allow
2. Planning-only tools (`TodoWrite`, `TaskCreate`, …) → always allow
3. `plan` mode → allow read-only tools; deny everything else
4. Read-only tools / read-only Bash → allow
5. Deny rules → deny
6. Allow rules → allow
7. Dangerous Bash → ask
8. Default → ask

**Simplified:** sandbox auto-allow path is omitted.

In [ ]:
PLAN_ALLOWED_TOOLS = {'Read', 'Grep', 'Glob'}
DANGEROUS_BASH_PREFIXES = [
    'rm ', 'sudo ', 'chmod ', 'chown ', 'mv ', 'dd ', 'mkfs',
    'shutdown', 'reboot', 'init 0', 'init 6',
    'git push', 'git reset --hard', 'git clean -fd',
]
READ_ONLY_BASH_FIRST_TOKENS = {
    'ls', 'cat', 'head', 'tail', 'grep', 'find', 'echo', 'pwd',
    'wc', 'sort', 'uniq', 'diff', 'stat', 'file', 'which',
}

@dataclass
class PermissionRuleSet:
    allow: list = field(default_factory=list)
    deny:  list = field(default_factory=list)

@dataclass
class PermissionRequest:
    tool_name: str
    input:     dict
    summary:   str
    risk:      str
    rule_hint: str

@dataclass
class PermissionResponse:
    behavior: str   # 'allow' | 'ask' | 'deny'
    reason:   str
    request:  PermissionRequest

# ── Rule matching ─────────────────────────────────────────────────────────

def _wildcard_re(pattern: str):
    parts = [re.escape(p) for p in pattern.split('*')]
    return re.compile('^' + '.*'.join(parts) + '$', re.IGNORECASE)

def matches_rule(rule: str, tool_name: str, input_: dict) -> bool:
    rule = rule.strip()
    if not rule:
        return False
    if rule == tool_name:
        return True
    if rule.startswith('mcp__') and '*' in rule:
        return bool(_wildcard_re(rule).match(tool_name))
    m = re.match(r'^([A-Za-z]+)\((.+)\)$', rule)
    if not m:
        return False
    rule_tool, pattern = m.group(1), m.group(2).strip()
    if rule_tool != tool_name:
        return False
    if tool_name == 'Bash':
        return bool(_wildcard_re(pattern).match(input_.get('command', '')))
    if tool_name == 'Skill':
        return bool(_wildcard_re(pattern).match(input_.get('skill', '')))
    return False

def matches_any(rules: list, tool: str, inp: dict) -> bool:
    return any(matches_rule(r, tool, inp) for r in rules)

# ── Helpers ───────────────────────────────────────────────────────────────

def is_dangerous_bash(cmd: str) -> bool:
    n = ' '.join(cmd.lower().split())
    return any(n.startswith(p) for p in DANGEROUS_BASH_PREFIXES)

def is_readonly_bash(cmd: str) -> bool:
    first = (cmd.strip().split() or [''])[0]
    return first in READ_ONLY_BASH_FIRST_TOKENS

def rule_hint(tool: str, inp: dict) -> str:
    if tool == 'Bash':
        first = (inp.get('command', '').split() or [''])[0]
        return f'Bash({first} *)' if first else 'Bash'
    if tool == 'Skill':
        return f"Skill({inp.get('skill', '')})"
    return tool

def risk_label(tool: str, is_readonly: bool, inp: dict) -> str:
    if tool == 'Bash':
        if is_dangerous_bash(inp.get('command', '')):
            return 'High risk: destructive shell command'
        if is_readonly_bash(inp.get('command', '')):
            return 'Low risk: read-only shell command'
        return 'Medium risk: shell command'
    return 'Low risk' if is_readonly else 'Medium risk: writes or side-effects'

# ── Main permission check ─────────────────────────────────────────────────

def check_permission(
    tool_name: str,
    is_readonly: bool,
    input_: dict,
    mode: str = 'default',
    settings: Optional[PermissionRuleSet] = None,
    session_rules: Optional[PermissionRuleSet] = None,
) -> PermissionResponse:
    if settings is None:      settings = PermissionRuleSet()
    if session_rules is None: session_rules = PermissionRuleSet()
    req = PermissionRequest(
        tool_name=tool_name, input=input_,
        summary=str(input_)[:80],
        risk=risk_label(tool_name, is_readonly, input_),
        rule_hint=rule_hint(tool_name, input_),
    )

    def resp(b, r): return PermissionResponse(b, r, req)

    if mode == 'auto':
        return resp('allow', 'auto mode allows all')

    if tool_name in ('TodoWrite', 'TaskCreate', 'TaskUpdate', 'TaskGet', 'TaskList'):
        return resp('allow', f'{tool_name} is planning-only state')

    if mode == 'plan':
        if tool_name in PLAN_ALLOWED_TOOLS: return resp('allow', 'read-only tool in plan mode')
        if tool_name in ('EnterPlanMode', 'ExitPlanMode'): return resp('ask', 'plan mode transition')
        if tool_name == 'Bash' and is_readonly_bash(input_.get('command', '')):
            return resp('allow', 'read-only bash in plan mode')
        return resp('deny', f'plan mode blocks {tool_name}')

    if tool_name == 'EnterPlanMode':
        return resp('ask', 'entering plan mode requires confirmation')

    if tool_name == 'Bash':
        if is_readonly_bash(input_.get('command', '')):
            return resp('allow', 'read-only shell command')
    elif is_readonly:
        return resp('allow', 'read-only tool')

    all_deny  = session_rules.deny  + settings.deny
    all_allow = session_rules.allow + settings.allow

    if matches_any(all_deny,  tool_name, input_): return resp('deny',  'matched deny rule')
    if matches_any(all_allow, tool_name, input_): return resp('allow', 'matched allow rule')

    if tool_name == 'Bash' and is_dangerous_bash(input_.get('command', '')):
        return resp('ask', 'dangerous bash requires confirmation')

    return resp('ask', 'operation requires confirmation')

# ── Smoke tests ───────────────────────────────────────────────────────────
for tool, ro, inp, mode in [
    ('Read',  True,  {},                         'default'),
    ('Bash',  False, {'command': 'ls -la'},       'default'),
    ('Bash',  False, {'command': 'rm -rf /tmp'},  'default'),
    ('Write', False, {'file_path': 'foo.txt'},    'plan'),
    ('Bash',  False, {'command': 'echo hi'},      'auto'),
]:
    r = check_permission(tool, ro, inp, mode)
    print(f'  {mode:8s} | {tool:8s} | {str(inp)[:35]:35s} => {r.behavior:6s}  {r.reason}')

## 4. Token Budget & Warning States

Inlined from `src/context/autoCompact.ts`.

The loop monitors estimated token usage every turn (after turn 1) and transitions
through four states:

```
normal → warning → error (auto-compact triggers) → blocking (loop halts)
```

**Simplified:** `estimate_tokens` uses char-count / 4 instead of the real
`tokenCountWithEstimation` anchor-based approach (`src/utils/tokens.ts`).

In [ ]:
MODEL_CONTEXT_WINDOWS = {
    'claude-opus-4-7':   200_000,
    'claude-sonnet-4-6': 200_000,
    'claude-haiku-4-5':  200_000,
}
DEFAULT_CONTEXT_WINDOW  = 200_000
AUTOCOMPACT_BUFFER      = 30_000
WARNING_BUFFER          = 40_000
MANUAL_COMPACT_BUFFER   = 10_000
SYSTEM_OUTPUT_RESERVE   = 20_000  # reserved for system prompt + output

def context_window(model: str) -> int:
    return MODEL_CONTEXT_WINDOWS.get(model, DEFAULT_CONTEXT_WINDOW)

def effective_window(model: str) -> int:
    return context_window(model) - SYSTEM_OUTPUT_RESERVE

def auto_compact_threshold(model: str) -> int:
    return max(0, effective_window(model) - AUTOCOMPACT_BUFFER)

def blocking_limit(model: str) -> int:
    return max(0, effective_window(model) - MANUAL_COMPACT_BUFFER)

def estimate_tokens(messages: list, system_prompt: str = '') -> int:
    # Rough approximation: 1 token ≈ 4 characters.
    # Real impl (tokens.ts) uses last-call usage as an anchor for accuracy.
    chars = len(system_prompt)
    for m in messages:
        content = m.content if isinstance(m.content, str) else json.dumps(m.content, default=str)
        chars += len(content)
    return chars // 4

@dataclass
class TokenWarningResult:
    state:            str   # 'normal' | 'warning' | 'error' | 'blocking'
    estimated_tokens: int
    threshold:        int
    blocking_limit:   int
    context_window:   int

def calculate_token_warning(estimated: int, model: str) -> TokenWarningResult:
    ctx      = context_window(model)
    eff      = effective_window(model)
    blocking = blocking_limit(model)
    compact  = auto_compact_threshold(model)
    warning  = max(0, eff - WARNING_BUFFER)

    if   estimated >= blocking: state = 'blocking'
    elif estimated >= compact:  state = 'error'
    elif estimated >= warning:  state = 'warning'
    else:                       state = 'normal'

    return TokenWarningResult(state, estimated, compact, blocking, ctx)

# Demo
model = 'claude-sonnet-4-6'
print(f'Context window: {context_window(model):,}   Effective: {effective_window(model):,}')
print(f'Warning at: {effective_window(model) - WARNING_BUFFER:,}   Auto-compact at: {auto_compact_threshold(model):,}   Blocking at: {blocking_limit(model):,}\n')
for est in [0, 100_000, 140_000, 160_000, 188_000]:
    w = calculate_token_warning(est, model)
    print(f'  {est:>8,} tokens  →  state={w.state}')

## 5. Streaming — Real Anthropic API Call

Mirrors `src/services/api/streaming.ts::streamMessage()`.

**Key implementation detail:** Tool input JSON is tracked **per content-block index**
(`tool_input_json_by_index`) because some providers emit `content_block_start` for
block N+1 before `content_block_stop` for block N. A shared buffer would corrupt inputs.

The function is an async generator:
- **Yields** `{type: 'text', text}`, `{type: 'tool_use_start', id, name}`, etc.
- **Returns** (as `StopAsyncIteration.value`) the assembled `StreamResult`.

In [ ]:
async def stream_message(
    messages:   list,
    model:      str  = 'claude-sonnet-4-6',
    system:     str  = '',
    tools:      Optional[list] = None,
    max_tokens: int  = 4096,
):
    """
    Async generator that mirrors streaming.ts::streamMessage().

    Yields incremental events; returns StreamResult dict when done.
    Caller should iterate with 'async for event in stream_message(...)'
    and capture the return value from 'async for' (it is the last yield).
    We embed the StreamResult as the final 'message_done' event to make
    consumption simpler in Python (no StopAsyncIteration gymnastics).
    """
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    # Convert MessageParam objects to Anthropic API dicts
    api_msgs = []
    for m in messages:
        if isinstance(m, MessageParam):
            if isinstance(m.content, str):
                api_msgs.append({'role': m.role, 'content': m.content})
            else:
                blocks = []
                for b in m.content:
                    if isinstance(b, ToolResultBlock):
                        d = {'type': 'tool_result', 'tool_use_id': b.tool_use_id, 'content': b.content}
                        if b.is_error: d['is_error'] = True
                        blocks.append(d)
                    elif isinstance(b, dict):
                        blocks.append(b)
                    else:
                        blocks.append(b.__dict__)
                api_msgs.append({'role': m.role, 'content': blocks})
        else:
            api_msgs.append(m)

    kwargs = dict(model=model, max_tokens=max_tokens, messages=api_msgs)
    if system: kwargs['system'] = system
    if tools:  kwargs['tools']  = tools

    # Per-index accumulators (the critical detail from streaming.ts)
    content_blocks: dict = {}          # index -> dict
    tool_input_json: dict = {}         # index -> accumulated JSON string
    usage    = Usage()
    stop_reason = ''

    with client.messages.stream(**kwargs) as stream:
        for event in stream:
            etype = event.type

            if etype == 'message_start':
                u = event.message.usage
                usage.input_tokens  = u.input_tokens
                usage.output_tokens = u.output_tokens

            elif etype == 'message_delta':
                usage.output_tokens = event.usage.output_tokens
                stop_reason = getattr(event.delta, 'stop_reason', '') or ''

            elif etype == 'content_block_start':
                idx = event.index
                cb  = event.content_block
                if cb.type == 'text':
                    content_blocks[idx] = {'type': 'text', 'text': ''}
                elif cb.type == 'tool_use':
                    content_blocks[idx] = {'type': 'tool_use', 'id': cb.id, 'name': cb.name, 'input': {}}
                    tool_input_json[idx] = ''
                    yield {'type': 'tool_use_start', 'id': cb.id, 'name': cb.name}

            elif etype == 'content_block_delta':
                idx   = event.index
                delta = event.delta
                if delta.type == 'text_delta':
                    content_blocks[idx]['text'] += delta.text
                    yield {'type': 'text', 'text': delta.text}
                elif delta.type == 'input_json_delta':
                    # Append to per-index buffer, NOT a shared one
                    tool_input_json[idx] = tool_input_json.get(idx, '') + delta.partial_json

            elif etype == 'content_block_stop':
                idx = event.index
                blk = content_blocks.get(idx)
                acc = tool_input_json.pop(idx, None)
                if blk and blk['type'] == 'tool_use' and acc:
                    try:
                        blk['input'] = json.loads(acc)
                    except json.JSONDecodeError:
                        blk['input'] = {'_raw': acc}   # keep raw string for debugging

    # Assemble final content blocks
    assembled = []
    for idx in sorted(content_blocks):
        b = content_blocks[idx]
        if b['type'] == 'text':
            assembled.append(TextBlock(text=b['text']))
        elif b['type'] == 'tool_use':
            assembled.append(ToolUseBlock(id=b['id'], name=b['name'], input=b['input']))

    # Yield final 'message_done' event (replaces the TS generator return value)
    yield {
        'type': 'message_done',
        'stop_reason': stop_reason,
        'usage': usage,
        'assembled': assembled,   # the completed content blocks
    }

print('stream_message() defined.')
print('Consume with: async for event in stream_message(...): ...')

## 6. Tool Registry (Stub)

In production, `src/tools/index.ts` exposes `findToolByName()` which looks up
a tool with `call()`, `isReadOnly()`, and `isConcurrencySafe()` methods.

Here we define two demo tools:
- **Calculator** — read-only, concurrency-safe (can run in parallel)
- **WriteFile** — side-effecting, serial (must run one-at-a-time)

In [ ]:
@dataclass
class ToolDefinition:
    name:             str
    description:      str
    readonly:         bool
    concurrency_safe: bool
    fn:               Any   # callable(input: dict) -> ToolResult

    def is_read_only(self) -> bool:
        return self.readonly

    def is_concurrency_safe(self, input_: dict = {}) -> bool:
        return self.concurrency_safe

    def call(self, input_: dict) -> ToolResult:
        return self.fn(input_)

    def to_api_param(self) -> dict:
        """Convert to Anthropic API tool format."""
        return {
            'name': self.name,
            'description': self.description,
            'input_schema': {'type': 'object', 'properties': {
                'expression': {'type': 'string', 'description': 'Math expression'}
            }, 'required': []},
        }

def _calculator(inp: dict) -> ToolResult:
    expr = inp.get('expression', '')
    if not re.match(r'^[\d\s\+\-\*\/\(\)\.]+$', expr):
        return ToolResult(content='Error: unsafe expression', is_error=True)
    try:
        return ToolResult(content=str(eval(expr)))
    except Exception as e:
        return ToolResult(content=f'Error: {e}', is_error=True)

def _write_file(inp: dict) -> ToolResult:
    p, c = inp.get('path', ''), inp.get('content', '')
    try:
        Path(p).write_text(c)
        return ToolResult(content=f'Wrote {len(c)} chars to {p}')
    except Exception as e:
        return ToolResult(content=f'Error: {e}', is_error=True)

TOOL_REGISTRY = {
    'Calculator': ToolDefinition('Calculator', 'Evaluates a math expression.',     True,  True,  _calculator),
    'WriteFile':  ToolDefinition('WriteFile',  'Writes text content to a file.',   False, False, _write_file),
}

def find_tool(name: str) -> Optional[ToolDefinition]:
    return TOOL_REGISTRY.get(name)

def get_tools_api_params() -> list:
    return [t.to_api_param() for t in TOOL_REGISTRY.values()]

print('Tools registered:', list(TOOL_REGISTRY.keys()))
print('Calculator 2+2 =>', _calculator({'expression': '2+2'}).content)

## 7. Tool Execution Pipeline

Inlined from `src/core/agenticLoop.ts`.

Three functions:

1. **`partition_tool_calls(blocks)`** — groups consecutive concurrency-safe blocks
   into parallel batches; non-safe blocks become singleton serial batches.
   ```
   [Calc, Calc, WriteFile, Calc]  →  [[Calc,Calc], [WriteFile], [Calc]]
   ```

2. **`run_one_tool_block(block)`** — permission check → `tool.call()` → truncate

3. **`run_tools(blocks)`** — iterates batches, runs parallel/serial, assembles
   `tool_result` message

**Simplified:** `on_permission_request` defaults to auto-deny for `ask` decisions.

In [ ]:
MAX_TOOL_USE_CONCURRENCY = 10   # mirrors agenticLoop.ts constant

@dataclass
class ToolExecutionResult:
    tool_use_id: str
    tool_name:   str
    tool_input:  dict
    result:      ToolResult

@dataclass
class ToolBatch:
    is_concurrency_safe: bool
    blocks: list = field(default_factory=list)

# ── 1. Partition ──────────────────────────────────────────────────────────

def partition_tool_calls(blocks: list) -> list:
    """
    Groups consecutive concurrency-safe blocks into parallel batches.
    Non-safe (or unknown) blocks become singleton serial batches.
    Mirrors agenticLoop.ts::partitionToolCalls().
    """
    batches = []
    for blk in blocks:
        name = blk.name if isinstance(blk, ToolUseBlock) else blk.get('name', '')
        tool = find_tool(name)
        safe = tool.is_concurrency_safe() if tool else False
        last = batches[-1] if batches else None
        if safe and last and last.is_concurrency_safe:
            last.blocks.append(blk)
        else:
            batches.append(ToolBatch(is_concurrency_safe=safe, blocks=[blk]))
    return batches

# ── 2. Run one block ──────────────────────────────────────────────────────

async def run_one_tool_block(
    blk,
    mode: str = 'default',
    session_rules: Optional[PermissionRuleSet] = None,
    on_permission_request=None,
) -> tuple:
    """Returns (ToolExecutionResult, PermissionRequest | None)."""
    name = blk.name  if isinstance(blk, ToolUseBlock) else blk.get('name', '')
    uid  = blk.id    if isinstance(blk, ToolUseBlock) else blk.get('id', '')
    inp  = blk.input if isinstance(blk, ToolUseBlock) else blk.get('input', {})

    tool = find_tool(name)
    if not tool:
        result = ToolResult(content=f'Error: Unknown tool "{name}"', is_error=True)
        return ToolExecutionResult(uid, name, inp, result), None

    perm = check_permission(
        tool_name=name, is_readonly=tool.is_read_only(),
        input_=inp, mode=mode, session_rules=session_rules,
    )

    if perm.behavior == 'deny':
        result = ToolResult(content=f'Permission denied for {name}: {perm.reason}', is_error=True)
        return ToolExecutionResult(uid, name, inp, result), None

    if perm.behavior == 'ask':
        decision = 'deny'
        if on_permission_request:
            decision = await on_permission_request(perm.request)
        if decision == 'deny':
            result = ToolResult(content=f'Permission denied for {name}.', is_error=True)
            return ToolExecutionResult(uid, name, inp, result), perm.request
        if decision == 'allow_always' and session_rules:
            session_rules.allow.append(perm.request.rule_hint)

    # ── Execute the tool ──────────────────────────────────────────────────
    raw    = tool.call(inp)
    # Truncate to 100k chars (mirrors Tool.ts::truncateToolResult)
    result = ToolResult(content=raw.content[:100_000], is_error=raw.is_error)
    return ToolExecutionResult(uid, name, inp, result), None

# ── 3. Run all tool blocks ────────────────────────────────────────────────

async def run_tools(
    content_blocks: list,
    mode: str = 'default',
    session_rules: Optional[PermissionRuleSet] = None,
    on_permission_request=None,
) -> dict:
    """
    Execute all tool_use blocks from an assistant message.
    Returns {tool_results_message, executions, permission_requests}.
    Mirrors agenticLoop.ts::runTools().
    """
    tool_use_blocks = [
        b for b in content_blocks
        if (isinstance(b, ToolUseBlock) and b.type == 'tool_use')
        or (isinstance(b, dict) and b.get('type') == 'tool_use')
    ]

    executions, perm_requests = [], []

    for batch in partition_tool_calls(tool_use_blocks):
        if batch.is_concurrency_safe and len(batch.blocks) > 1:
            # Parallel — Promise.all equivalent
            results = await asyncio.gather(*[
                run_one_tool_block(b, mode, session_rules, on_permission_request)
                for b in batch.blocks
            ])
        else:
            # Serial — side-effecting tools must run one-at-a-time
            results = []
            for b in batch.blocks:
                results.append(await run_one_tool_block(b, mode, session_rules, on_permission_request))

        for exec_result, perm_req in results:
            executions.append(exec_result)
            if perm_req: perm_requests.append(perm_req)

    # Build tool_result message in original block order
    tool_results = []
    for e in executions:
        tr = {'type': 'tool_result', 'tool_use_id': e.tool_use_id, 'content': e.result.content}
        if e.result.is_error: tr['is_error'] = True
        tool_results.append(tr)

    return {
        'tool_results_message': MessageParam(role='user', content=tool_results),
        'executions':           executions,
        'permission_requests':  perm_requests,
    }

# ── Demo: partition logic ─────────────────────────────────────────────────
demo_blocks = [
    ToolUseBlock(id='t1', name='Calculator', input={'expression': '2+2'}),
    ToolUseBlock(id='t2', name='Calculator', input={'expression': '3*3'}),
    ToolUseBlock(id='t3', name='WriteFile',  input={'path': '/tmp/x.txt', 'content': 'hi'}),
    ToolUseBlock(id='t4', name='Calculator', input={'expression': '10/2'}),
]
print('Partition demo:')
for i, b in enumerate(partition_tool_calls(demo_blocks)):
    names = [bl.name for bl in b.blocks]
    print(f'  Batch {i+1}: parallel={b.is_concurrency_safe}  tools={names}')

## 8. The `query()` Generator — Core Agentic Loop

Inlined from `src/core/agenticLoop.ts::query()`.

This is the **heartbeat of agency**: an `async def` generator that implements
Reason → Act → Observe until the model stops calling tools.

| Source decision | Source location | Notebook |
|---|---|---|
| Max turns guard | `agenticLoop.ts:450` | `while turn_count < max_turns` |
| Abort check | `agenticLoop.ts:451` | `if abort_signal.is_set()` |
| Token budget check | `agenticLoop.ts:460-483` | `calculate_token_warning()` |
| Stream → accumulate | `agenticLoop.ts:497-539` | `async for event in stream_message(...)` |
| Continuation decision | `agenticLoop.ts:563` | `if stop_reason != 'tool_use': return` |
| Run tools | `agenticLoop.ts:568` | `await run_tools(...)` |
| Observe | `agenticLoop.ts:597-602` | append `tool_results_message` to `state.messages` |

In [ ]:
MAX_TOOL_TURNS = 50   # hard cap — mirrors agenticLoop.ts::MAX_TOOL_TURNS

async def query(
    messages:         list,
    model:            str  = 'claude-sonnet-4-6',
    system_prompt:    str  = '',
    tools_api:        Optional[list] = None,
    max_turns:        int  = MAX_TOOL_TURNS,
    abort_signal:     Optional[asyncio.Event] = None,
    mode:             str  = 'default',
    session_rules:    Optional[PermissionRuleSet] = None,
    on_permission_request = None,
):
    """
    Async generator — the Reason → Act → Observe loop.

    Yields event dicts:
      {type: 'text', text}
      {type: 'tool_use_start', id, name}
      {type: 'tool_use_done', id, name, input, result}
      {type: 'assistant_message', message}
      {type: 'tool_result_message', message}
      {type: 'token_warning', warning}
      {type: 'turn_usage', turn_usage, cumulative_usage, turn_count}
      {type: 'turn_complete', reason, turn_count}
      {type: 'error', error}

    The final 'turn_complete' event carries reason:
      'completed'      — model produced end_turn (no more tool calls)
      'aborted'        — abort_signal fired
      'max_turns'      — hit MAX_TOOL_TURNS
      'blocking_limit' — token budget exhausted
      'model_error'    — stream returned nothing or errored
    """
    state        = LoopState(messages=list(messages))
    total_usage  = Usage()
    last_usage   = Usage()

    while state.turn_count < max_turns:

        # ── Guard: abort ─────────────────────────────────────────────────
        if abort_signal and abort_signal.is_set():
            yield {'type': 'turn_complete', 'reason': 'aborted', 'turn_count': state.turn_count}
            return

        next_turn = state.turn_count + 1

        # ── Guard: token budget (skip on first turn) ─────────────────────
        # The first turn is skipped because we have no usage anchor yet.
        if state.turn_count > 0:
            est     = estimate_tokens(state.messages, system_prompt)
            warning = calculate_token_warning(est, model)
            if warning.state != 'normal':
                yield {'type': 'token_warning', 'warning': warning}
            if warning.state == 'blocking':
                yield {'type': 'error',        'error': f'Context window limit reached ({est} tokens)'}
                yield {'type': 'turn_complete', 'reason': 'blocking_limit', 'turn_count': next_turn}
                return

        # ── REASON: stream from LLM ──────────────────────────────────────
        assembled = []
        stop_reason = ''

        async for event in stream_message(
            messages=state.messages, model=model,
            system=system_prompt, tools=tools_api,
        ):
            if event['type'] == 'text':
                yield event
            elif event['type'] == 'tool_use_start':
                yield event
            elif event['type'] == 'message_done':
                stop_reason   = event['stop_reason']
                last_usage    = event['usage']
                assembled     = event.get('assembled', [])
                total_usage.input_tokens  += last_usage.input_tokens
                total_usage.output_tokens += last_usage.output_tokens

        # ── Build assistant message & update state ────────────────────────
        assistant_msg = MessageParam(role='assistant', content=assembled or 'no content')
        state.messages.append(assistant_msg)
        state.turn_count = next_turn
        yield {'type': 'assistant_message', 'message': assistant_msg}
        yield {'type': 'turn_usage', 'turn_usage': last_usage,
               'cumulative_usage': total_usage, 'turn_count': state.turn_count}

        # ── CONTINUATION DECISION ─────────────────────────────────────────
        # The model itself decides whether to continue or stop.
        # stop_reason == 'tool_use'  → model wants to call more tools → keep going
        # stop_reason == 'end_turn'  → model is done → terminate
        if stop_reason != 'tool_use':
            yield {'type': 'turn_complete', 'reason': 'completed', 'turn_count': state.turn_count}
            return

        # ── ACT: execute tool calls ───────────────────────────────────────
        run_result = await run_tools(
            assembled, mode=mode,
            session_rules=session_rules,
            on_permission_request=on_permission_request,
        )

        for req in run_result['permission_requests']:
            yield {'type': 'permission_request', 'request': req}

        for ex in run_result['executions']:
            yield {'type': 'tool_use_done', 'id': ex.tool_use_id,
                   'name': ex.tool_name, 'input': ex.tool_input, 'result': ex.result}

        # ── OBSERVE: feed tool results back ───────────────────────────────
        # Appending to state.messages is what gives the agent its 'memory'
        # within a single query — each result informs the next reasoning step.
        state.messages.append(run_result['tool_results_message'])
        yield {'type': 'tool_result_message', 'message': run_result['tool_results_message']}

    # Max turns exhausted
    yield {'type': 'turn_complete', 'reason': 'max_turns', 'turn_count': state.turn_count}

print('query() generator defined.')

## 9. `QueryEngine` — Session-Level Orchestrator

Inlined from `src/core/queryEngine.ts`.

Wraps `query()` with:
- Persistent `messages` across multiple user turns
- Slash command dispatch (`/clear`, `/mode`, `/cost`, `/model`)
- Permission mode management

**Simplified:** compaction hooks are stubs; MCP / history / skill commands are omitted.

In [ ]:
class QueryEngine:
    """
    Session-level wrapper around query().
    Mirrors src/core/queryEngine.ts::QueryEngine.
    """
    def __init__(
        self,
        model:         str = 'claude-sonnet-4-6',
        system_prompt: str = '',
        mode:          str = 'default',
        session_rules: Optional[PermissionRuleSet] = None,
    ):
        self.messages:               list                   = []
        self.total_usage:            Usage                  = Usage()
        self.last_call_usage:        Usage                  = Usage()
        self.model:                  str                    = model
        self.session_model_override: Optional[str]          = None
        self.system_prompt:          str                    = system_prompt
        self.mode:                   str                    = mode
        self.session_rules:          PermissionRuleSet      = session_rules or PermissionRuleSet()
        self._abort_event:           Optional[asyncio.Event] = None

    @property
    def active_model(self) -> str:
        return self.session_model_override or self.model

    def interrupt(self) -> bool:
        if self._abort_event:
            self._abort_event.set()
            return True
        return False

    async def submit_message(self, user_input: str):
        """
        Handle one user turn. Async generator yielding QueryEngineEvent dicts.
        Slash commands are intercepted before reaching the model.
        """
        trimmed = user_input.strip()
        if not trimmed:
            return

        if trimmed.startswith('/'):
            async for event in self._handle_command(trimmed):
                yield event
            return

        # Append user message to persistent history
        self.messages.append(MessageParam(role='user', content=trimmed))
        yield {'type': 'messages_updated', 'messages': list(self.messages)}

        self._abort_event = asyncio.Event()
        try:
            async for event in query(
                messages=list(self.messages),
                model=self.active_model,
                system_prompt=self.system_prompt,
                tools_api=get_tools_api_params(),
                abort_signal=self._abort_event,
                mode=self.mode,
                session_rules=self.session_rules,
            ):
                yield event
                # Mirror QueryEngine's live message sync
                if event['type'] in ('assistant_message', 'tool_result_message'):
                    self.messages.append(event['message'])
                    yield {'type': 'messages_updated', 'messages': list(self.messages)}
                if event['type'] == 'turn_usage':
                    self.last_call_usage = event['turn_usage']
        finally:
            self._abort_event = None

    async def _handle_command(self, command: str):
        """Slash command dispatcher — mirrors QueryEngine.handleCommand()."""
        parts = command[1:].split()
        name, args = (parts[0] if parts else ''), parts[1:]

        if name == 'help':
            yield {'type': 'command', 'kind': 'info',
                   'message': 'Commands: /help /clear /cost /model [name|default] /mode [default|plan|auto]'}

        elif name == 'clear':
            self.messages = []
            yield {'type': 'session_cleared'}
            yield {'type': 'messages_updated', 'messages': []}
            yield {'type': 'command', 'kind': 'info', 'message': 'Conversation cleared.'}

        elif name == 'cost':
            yield {'type': 'command', 'kind': 'info',
                   'message': f'Input: {self.total_usage.input_tokens}  Output: {self.total_usage.output_tokens}'}

        elif name == 'model':
            next_m = ' '.join(args).strip()
            if not next_m:
                yield {'type': 'command', 'kind': 'info', 'message': f'Active model: {self.active_model}'}
            elif next_m == 'default':
                self.session_model_override = None
                yield {'type': 'model_changed', 'model': self.active_model, 'source': 'default'}
            else:
                self.session_model_override = next_m
                yield {'type': 'model_changed', 'model': next_m, 'source': 'session'}

        elif name == 'mode':
            next_mode = args[0] if args else ''
            if next_mode in ('default', 'plan', 'auto'):
                prev, self.mode = self.mode, next_mode
                yield {'type': 'mode_changed', 'mode': self.mode, 'previous': prev}
                yield {'type': 'command', 'kind': 'info', 'message': f'Mode: {prev} → {self.mode}'}
            elif next_mode:
                yield {'type': 'command', 'kind': 'error', 'message': f'Unknown mode: {next_mode}'}
            else:
                yield {'type': 'command', 'kind': 'info', 'message': f'Current mode: {self.mode}'}

        else:
            yield {'type': 'command', 'kind': 'error', 'message': f'Unknown command: /{name}'}

print('QueryEngine defined.')

## 10. Live Demo — Full Agentic Session

Runs a real Anthropic API session. The model will use the **Calculator** tool.

Watch the event sequence:
1. `text` — model starts reasoning
2. `tool_use_start` — model decides to call Calculator
3. `tool_use_done` — Calculator result returned
4. `text` — model reads result and produces final answer
5. `turn_complete` reason=`completed`

In [ ]:
async def demo_session():
    engine = QueryEngine(
        model='claude-sonnet-4-6',
        system_prompt=(
            'You are a helpful assistant with access to a Calculator tool. '
            'When the user asks for a calculation, use the Calculator tool.'
        ),
    )

    query_text = 'What is 47 * 89 + 123? Use the Calculator tool.'
    print(f'User: {query_text}\n')

    async for event in engine.submit_message(query_text):
        t = event.get('type', '')
        if t == 'text':
            print(event['text'], end='', flush=True)
        elif t == 'tool_use_start':
            print(f'\n[→ Tool call: {event["name"]} (id={event["id"]})]')
        elif t == 'tool_use_done':
            r = event['result']
            status = 'ERROR' if r.is_error else 'OK'
            print(f'[← Tool result ({status}): {r.content}]')
        elif t == 'turn_usage':
            u = event['turn_usage']
            print(f'\n[Turn {event["turn_count"]} usage: in={u.input_tokens} out={u.output_tokens}]')
        elif t == 'turn_complete':
            print(f'[Loop ended: reason={event["reason"]}]')
        elif t == 'token_warning':
            print(f'[Token warning: {event["warning"].state}]')
        elif t == 'permission_request':
            print(f'[Permission requested for {event["request"].tool_name} — auto-denied]')
        # Suppress messages_updated — noisy

if ANTHROPIC_API_KEY:
    await demo_session()
else:
    print('Skipped — ANTHROPIC_API_KEY not set.')

## 11. Slash Commands & Permission Mode

Demonstrates `QueryEngine`'s command dispatch — no API calls.

In [ ]:
async def demo_commands():
    engine = QueryEngine()
    cmds = [
        '/help',
        '/cost',
        '/model claude-opus-4-7',
        '/model',
        '/model default',
        '/mode auto',
        '/mode',
        '/clear',
        '/mode invalid',
        '/unknown',
    ]
    for cmd in cmds:
        print(f'>> {cmd}')
        async for event in engine.submit_message(cmd):
            t = event.get('type', '')
            if t == 'command':
                prefix = '[ERROR]' if event['kind'] == 'error' else '[INFO] '
                # Print only first line so output stays compact
                print(f'   {prefix} {event["message"].splitlines()[0]}')
            elif t == 'model_changed':
                print(f'   [MODEL] -> {event["model"]} ({event["source"]})')
            elif t == 'mode_changed':
                print(f'   [MODE]  {event["previous"]} -> {event["mode"]}')
            elif t == 'session_cleared':
                print('   [SESSION CLEARED]')
        print()

await demo_commands()

## 12. Summary

| Notebook cell | TypeScript source | Core concept |
|---|---|---|
| 2. Data Models | `src/types/message.ts` | `TextBlock`, `ToolUseBlock`, `Usage`, `MessageParam` |
| 3. Permission System | `src/permissions/permissions.ts` | Layered allow/deny/ask decision tree |
| 4. Token Budget | `src/context/autoCompact.ts` | normal→warning→error→blocking states |
| 5. Streaming | `src/services/api/streaming.ts` | Per-index block assembly, delta accumulation |
| 6. Tool Registry | `src/tools/index.ts` | `findToolByName()`, `isConcurrencySafe()` |
| 7. Tool Pipeline | `agenticLoop.ts` `runTools()` | `partitionToolCalls`, parallel vs serial batches |
| 8. `query()` Loop | `agenticLoop.ts` `query()` | The Reason→Act→Observe async generator |
| 9. `QueryEngine` | `src/core/queryEngine.ts` | Session state, compaction, slash commands |

### The single most important line

```python
if stop_reason != 'tool_use':
    return   # model is done — loop terminates
```

When `stop_reason == 'tool_use'` the model wants to keep going.
The loop runs the tools, feeds results back as a new `user` message,
and calls the LLM again. When it's `end_turn`, the task is complete.

**The model controls its own execution flow. The loop is just the executor.**